# **04.텍스트 분류 실습 - 20 뉴스그룹 분류**
* fetch_20newsgroups() api 이용해 뉴스그룹의 분류 수행

**텍스트 정규화**

In [ ]:
from sklearn.datasets import fetch_20newsgroups

news_data = fetch_20newsgroups(subset = 'all', random_state =156)

In [ ]:
print(news_data.keys())

dict_keys(['data', 'filenames', 'target_names', 'target', 'DESCR'])


* filenames라는 key => 로컬 컴퓨터에 저장하는 디렉터리와 파일명 지칭

In [ ]:
import pandas as pd

print('target 클래스의 값과 분포도 \n', pd.Series(news_data.target).value_counts().sort_index())
print('target 클래스의 이름들 \n', news_data.target_names)

target 클래스의 값과 분포도 
 0     799
1     973
2     985
3     982
4     963
5     988
6     975
7     990
8     996
9     994
10    999
11    991
12    984
13    990
14    987
15    997
16    910
17    940
18    775
19    628
Name: count, dtype: int64
target 클래스의 이름들 
 ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']


In [ ]:
print(news_data.data[0])

From: egreen@east.sun.com (Ed Green - Pixel Cruncher)
Subject: Re: Observation re: helmets
Organization: Sun Microsystems, RTP, NC
Lines: 21
Distribution: world
Reply-To: egreen@east.sun.com
NNTP-Posting-Host: laser.east.sun.com

In article 211353@mavenry.altcit.eskimo.com, maven@mavenry.altcit.eskimo.com (Norman Hamer) writes:
> 
> The question for the day is re: passenger helmets, if you don't know for 
>certain who's gonna ride with you (like say you meet them at a .... church 
>meeting, yeah, that's the ticket)... What are some guidelines? Should I just 
>pick up another shoei in my size to have a backup helmet (XL), or should I 
>maybe get an inexpensive one of a smaller size to accomodate my likely 
>passenger? 

If your primary concern is protecting the passenger in the event of a
crash, have him or her fitted for a helmet that is their size.  If your
primary concern is complying with stupid helmet laws, carry a real big
spare (you can put a big or small head in a big helmet, bu

* 내용을 제외하고 제목 등의 다른 정보 제거.=>헤더와 푸터 정보들은 target 클래스 값과 유사한 데이터 가지고 있기 때문

In [ ]:
from sklearn.datasets import fetch_20newsgroups

train_news = fetch_20newsgroups(subset = 'train', remove = ('headers', 'footers', 'quotes'), random_state = 156)
X_train = train_news.data
y_train = train_news.target

test_news = fetch_20newsgroups(subset = 'test', remove = ('headers', 'footers', 'quotes'), random_state = 156)
X_test = test_news.data
y_test = test_news.target
print('학습 데이터 크기 {0}, 테스트 데이터 크기 {1}'.format(len(train_news.data), len(test_news.data)))

학습 데이터 크기 11314, 테스트 데이터 크기 7532


**피치 벡터화 변환과 머신러닝 모델 학습/예측/평가**

* 학습데이터: 11314개의 뉴스그룹 문서가 리스트 형태로 주어짐
* 테스트 데이터: 7532개의 문서가 역시 리스트 형태로 주어짐
* CountVectorizer 이용해 학습 데이터의 텍스트를 피처 벡터화
* 테스트 데이터에서 CountVectorizer를 적용 시, 반드시 학습 데이터를 이용해 fit()이 수행된 객체를 이용해 테스트 데이터 변환(trainsform) => 피처 개수가 같아야함
* 단, fit_trainsform()을 사용X => 테스트 데이터 기반으로 다시 CountVectorizer 가 fit()을 수행하고 trainsfomr()하기에 학습시 사용된 피처 개수와 예측 시 사용할 피처 개수 달라짐

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

#feature vectorization
cnt_vect = CountVectorizer()
cnt_vect.fit(X_train)
X_train_cnt_vect = cnt_vect.transform(X_train)

#transform
X_test_cnt_vect = cnt_vect.transform(X_test)

print('학습 데이터 텍스트의 CountVectorizer Shape:', X_train_cnt_vect.shape)

학습 데이터 텍스트의 CountVectorizer Shape: (11314, 101631)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

lr_clf = LogisticRegression()
lr_clf.fit(X_train_cnt_vect, y_train)
pred = lr_clf.predict(X_test_cnt_vect)
print('CounVectorized Logistic Regression의 예측 정확도는 {0:.3f}'.format(accuracy_score(y_test, pred)))

CounVectorized Logistic Regression의 예측 정확도는 0.604


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vect = TfidfVectorizer()
tfidf_vect.fit(X_train)
X_train_tfidf_vect = tfidf_vect.transform(X_train)
X_test_tfidf_vect = tfidf_vect.transform(X_test)

lr_clf = LogisticRegression()
lr_clf.fit(X_train_tfidf_vect, y_train)
pred = lr_clf.predict(X_test_tfidf_vect)
print('TF-IDF Logistic Regression의 예측 정확도는 {0:.3f}'.format(accuracy_score(y_test, pred)))

TF-IDF Logistic Regression의 예측 정확도는 0.674


* TF-IDF가 단순 카운트 기반보다 더 높은 예측 정확도 제공

**모델 성능 향상 방법**
1. 최적의 ML 알고리즘 선택
2. 최상의 피처 전처리 수행

In [ ]:
#stop words 필터리 추가. ngram을 기본 (1,1)에서 (1,2)로 변경해 피처 벡터화 적용
tfidf_vect = TfidfVectorizer(stop_words = 'english', ngram_range=(1,2), max_df = 300)
tfidf_vect.fit(X_train)
X_train_tfidf_vect = tfidf_vect.transform(X_train)
X_test_tfidf_vect = tfidf_vect.transform(X_test)

lr_clf = LogisticRegression(solver = 'liblinear')
lr_clf.fit(X_train_tfidf_vect, y_train)
pred = lr_clf.predict(X_test_tfidf_vect)
print('TF-IDF Vectorized Logistic Regression의 예측 정확도는 {0:.3f}'.format(accuracy_score(y_test, pred)))

TF-IDF Vectorized Logistic Regression의 예측 정확도는 0.690


In [ ]:
from sklearn.model_selection import GridSearchCV

params = {'C':[0.01, 0.1, 1, 5, 10]}
grid_cv_lr = GridSearchCV(lr_clf, param_grid=params, scoring='accuracy', verbose = 1)
grid_cv_lr.fit(X_train_tfidf_vect, y_train)
print('Logistic Regression best C parameter:', grid_cv_lr.best_params_)

pred = grid_cv_lr.predict(X_test_tfidf_vect)
print('TF-IDF Vectorized Logistic Regression의 예측 정확도는 {0:.3f}'.format(accuracy_score(y_test, pred)))


Fitting 5 folds for each of 5 candidates, totalling 25 fits
Logistic Regression best C parameter: {'C': 10}
TF-IDF Vectorized Logistic Regression의 예측 정확도는 0.704


**사이킷런 파이프라인 사용 및 GridSearchCV와의 결합**
* 파이프라인 이용시 피처 벡터화 + 알고리즘 학습/예측 코드 작성 한번에 진행
* 데이터 전처리와 ML 학습 과정을 통일된 API 기반에서 처리 가능
* 대용량 데이터의 피처 벡터화 결과를 별도의 데이터로 저장X. 스트림 기반에서 바로 데이터로 입력할 수 있음 => 모든 전처리 작업과 Estimator결합

In [ ]:
from sklearn.pipeline import Pipeline
pipeline = Pipeline([
    ('tfidf_vect', TfidfVectorizer(stop_words = 'english')),
    ('lr_clf', LogisticRegression(random_state=156))
])

* TfidfVectorizer 객체를 tfidCvect라는 객체 변수명으로, LogisticRegression 객체를 lr_clf라
는 객체 변수명으로 생성한 뒤 이 두 개의 객체를 파이프라인으로 연결하는 Pip이ine 객체 pipeline을
생성한다는 의미
* 기존 TfidfVectorizer의 학습 데이터와 테스트 데이터
에 대한 fit()과 transform( ) 수행을 통한 피처 벡터화와 LogisticRegressor의 fit()과 predict() 수
행을 통한 머신러닝 모델의 학습과 예측이 Pipeline의 fit()과 predict()로 통일돼 수행됨

In [ ]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('tfidf_vect', TfidfVectorizer(stop_words = 'english', ngram_range=(1,2), max_df = 300)),
    ('lr_clf', LogisticRegression(solver = 'liblinear', C = 10))
])

pipeline.fit(X_train, y_train)
pred = pipeline.predict(X_test)
print('Pipeline을 통한 Logistic Regression의 예측 정확도는 {0:.3f}'.format(accuracy_score(y_test, pred)))

Pipeline을 통한 Logistic Regression의 예측 정확도는 0.704


* GridSearchCV에 Pipeline을 입력하면서 TfidfVectorizer의 파라미터와 Logistic
Regression의 하이퍼 파라미터를 함께 최적화
* Pipeline을 입력할 경우에는 param_grid의 입력값 설정이 기존과 약간 다름
*  tfidf_vect__ngram_range’와 같이 하이퍼 파라미터명이 객체 변수명과 결합
* tfdifLyect의 ngram_range 파라미터 값을 변화시키면서 최적화하기를
원한다면 객체 변수명인 tfidf_vect에 언더바2개를 연달아붙인 뒤 파라미터명인 ngram_range를
결합해 ‘tfidf_vect__ngram_range’를 Key 값으로 할당
* Pipeline + GridSearchCV를 적용할 때 유의할 점은 모두의 파라미터를 최적화하려면 너무 많은 튜
닝 시간이 소모

In [ ]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('tfidf_vect', TfidfVectorizer(stop_words = 'english')),
    ('lr_clf', LogisticRegression())
])

params = { 'tfidf_vect__ngram_range':[(1,1), (1,2), (1,3)],
          'tfidf_vect__max_df': [100, 300, 700],
           'lr_clf__C':[1,5,10]}

grid_cv_pipe = GridSearchCV(pipeline, param_grid = params, cv = 3, scoring = 'accuracy', verbose = 1)
grid_cv_pipe.fit(X_train, y_train)
grid_cv_pipe.fit(X_train, y_train)
print(grid_cv_pipe.best_params_, grid_cv_pipe.best_score_)

pred = grid_cv_pipe.predict(X_test)
print('Pipeline을 통한 Logistic Regression의 예측 정확도는 {0:.3f}'.format(accuracy_score(y_test, pred)))

Fitting 3 folds for each of 27 candidates, totalling 81 fits
